In [4]:
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional
from sentence_transformers import SentenceTransformer
import chromadb
import fitz

In [5]:

@dataclass
class Document:
    page_content : str
    metadata: Dict[str ,Any]
    
class DocumentProcessor:

    def load_pdf(self, pdf_path):
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            page_text = page.get_text()
            if page_text:
                text += page_text+'\n'
        doc.close()
        return text

    def clean_text(self, text):
        text = text.replace('\00', " ")
        lines = [line.strip() for line in text.splitlines()]
        paragraphs = []
        buffer = []
        for line in lines:
            if line:
                buffer.append(line)
            elif buffer:
                paragraphs.append(" ".join(buffer))
                buffer = []  
        if buffer:
            paragraphs.append(" ".join(buffer))
            
        return "\n\n".join(paragraphs)

    def chunk_text(self, text,chunk_size=500, overlap=100):
        chunks = []
        start = 0
        while(start < len(text)):
            end = start + chunk_size
            chunks.append(text[start:end])
            start += chunk_size - overlap
        return chunks
    

    
class EmbeddingManager():
    def __init__(self, model_name):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        
    def embed_documents(self, chunks):
        return self.model.encode(chunks).tolist()
    def embed_query(self, query):
        return self.model.encode(query).tolist()
    
class VectorStore():
    def __init__(self):
        client = chromadb.PersistentClient(path = "../data/ragtrial")
        self.collection = client.get_or_create_collection("documents")
        
    def add_documents(self, chunks, embeddings):
        self.collection.add(
            documents = chunks,
            embeddings= embeddings,
            ids = [str(i) for i in range(len(chunks))]
        )
        
    def search(self, query_embedding, k = 5):
        return self.collection.query(
            query_embeddings=[query_embedding],
            n_results=k
        )

class RAGPipeline():
    def __init__(self):
        self.processor = DocumentProcessor()
        self.embedding = EmbeddingManager("all-MiniLM-L6-v2")
        self.vectorstore = VectorStore()
        
    def index_pdf(self, pdf_path):
        text = self.processor.load_pdf(pdf_path)
        text = self.processor.clean_text(text)
        chunks = self.processor.chunk_text(text)
        embeddings = self.embedding.embed_documents(chunks)
        self.vectorstore.add_documents(chunks, embeddings)
        
    def ask(self, question):
        query_embedding = self.embedding.embed_query(question)
        results = self.vectorstore.search(query_embedding)
        return results

In [6]:
rag = RAGPipeline()
chunks = rag.index_pdf("D:\\NLP\\RAG\\data\\pdfs\\Institute Receipt.pdf")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11863.61it/s]


In [7]:
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage
import os

In [9]:
llm = ChatMistralAI(
    model = os.getenv("MISTRAL_MODEL", "mistral-small-latest"),
    temperature= 0.5
)
prompt = input("Ask any thing related to the cotext !!!")
context = rag.ask(f"{prompt}")
context = context['documents'][0][0]
final_prompt = f"From the context {context} you need to answer {prompt} if it is out of context if you know answer you can say if you not have any prior knowledge say `This thing is out of context`" 
response = llm.invoke(final_prompt)
print(response.content)

Hello
